# Naive Bayes Classifier for Congressional Voting Records

This notebook demonstrates Naive Bayes solution for the Congressional Voting Records dataset.

## Highlights
1. **Manual Stratified Splits** (80:20 train-test, plus 10-fold CV).
2. **Two modes for missing values** (`?` treated as a third category or replaced with column mode).
3. **Laplace Smoothing** parameter \(\lambda\) for controlling the smoothing degree.
4. **Use of Logarithms** to avoid numerical underflow.
5. **Mean Accuracy and Standard Deviation** across the folds.

At the end, we compare train-set performance, 10-fold cross-validation metrics, and final test-set accuracy.


In [6]:
!pip install ucimlrepo

In [7]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import random
import statistics
from math import log

## 1. Fetch and Prepare the Dataset
We fetch the Congressional Voting Records (UCI ID=105) using `ucimlrepo`. The dataset has 16 features and 1 target (`Class`, which is "democrat" or "republican").

In [43]:
# Fetch dataset
congressional_voting_records = fetch_ucirepo(id=105)

# The data is returned as Pandas DataFrames
X = congressional_voting_records.data.features  # Features
y = congressional_voting_records.data.targets   # Target

# Merge features and target into a single DataFrame
df = pd.concat([X, y], axis=1)

# Display the first few rows
df.head()

,handicapped-infants,water-project-cost-sharing,adoption-of-the-budget-resolution,physician-fee-freeze,el-salvador-aid,religious-groups-in-schools,anti-satellite-test-ban,aid-to-nicaraguan-contras,mx-missile,immigration,synfuels-corporation-cutback,education-spending,superfund-right-to-sue,crime,duty-free-exports,export-administration-act-south-africa,Class
0,n,y,n,y,y,y,n,n,n,y,NaN,y,y,y,n,y,republican
1,n,y,n,y,y,y,n,n,n,n,n,y,y,y,n,NaN,republican
2,NaN,y,y,NaN,y,y,n,n,n,n,y,n,y,y,n,n,democrat
3,n,y,y,n,NaN,y,n,n,n,n,y,n,y,n,n,y,democrat
4,y,y,y,n,y,y,n,n,n,n,y,NaN,y,y,y,y,democrat


We extract the feature names (all but the last column) and store unique values for each feature in a global dictionary `feature_values`. This helps us handle the different possible feature values (including `?`).

In [44]:
# Feature list
features = list(df.columns)[:-1]  # all columns except 'Class'

# Store possible feature values in a dictionary.
feature_values = {feat: df[feat].unique() for feat in features}

## 2. Functions to Handle Missing Values and Data Splitting
We provide two methods for missing values (`?`):
- **Option 0**: Treat `?` as a valid third option (like "abstain").
- **Option 1**: Replace `?` with the most frequent value in the corresponding column.

### Data Splitting (Train/Test)
We manually perform a stratified split of 80% for training and 20% for testing, ensuring the class ratio is preserved.

In [45]:
def get_complete_data(data, mode=0):
    """
    Processes missing values ('?') in the dataset.
    mode=0 -> '?' is treated as a third valid category.
    mode=1 -> '?' is replaced by the most frequent value of that column.
    Returns a new DataFrame.
    """
    result = data.copy(deep=True)

    if not mode == 0:
        # Replace '?' with the column's mode (most frequent)
        for column in result.columns:
            if column == "Class":
                continue  # skip the target column
            col_mode = result.loc[result[column] != "?", column].mode()
            if len(col_mode) > 0:
                most_freq = col_mode.iloc[0]
                result.loc[result[column] == "?", column] = most_freq

    # Update feature_values with the final possible values
    feature_values = {feat: result[feat].unique() for feat in features}

    return result, feature_values

### Manual Stratified Split (80:20)
Below, we:
1. Separate the dataset by class.
2. Shuffle each class separately.
3. Take 80% from each class for training, 20% for testing.
4. Combine them back into `train_df` and `test_df`.

This preserves the proportion of classes in both subsets.

In [46]:
def stratified_train_test_split(data, test_ratio=0.2):
    """
    Manually performs a stratified split by class.
    test_ratio is 0.2 -> 80% train, 20% test.
    """
    data = data.sample(frac=1).reset_index(drop=True)  # shuffle overall
    train_records = []
    test_records = []

    classes = data['Class'].unique()
    # For each class, we do separate splits
    for c in classes:
        c_data = data[data['Class'] == c]
        c_data = c_data.sample(frac=1).reset_index(drop=True)
        n = len(c_data)
        split_idx = int((1 - test_ratio) * n)

        train_records.extend(c_data.iloc[:split_idx].to_dict('records'))
        test_records.extend(c_data.iloc[split_idx:].to_dict('records'))

    train_df = pd.DataFrame(train_records).reset_index(drop=True)
    test_df = pd.DataFrame(test_records).reset_index(drop=True)
    return train_df, test_df

## 3. Naive Bayes Model Implementation
We will:
1. Calculate **class priors** \(P(\text{class})\) using counts.
2. Compute **conditional probabilities** \(P(\text{feature_value} | \text{class})\) with **Laplace smoothing**.
3. Use the **log** of the probabilities to avoid numerical underflow.
4. Make a prediction by summing all log probabilities and selecting the class with the highest sum.
5. Estimate accuracy by comparing predicted labels with actual labels.

In [47]:
def get_class_info(data, laplace_lambda=0.1):
    """
    For each class, compute:
      - class_title: the actual class name (democrat/republican)
      - class_table: subset of data belonging to that class
      - class_prob: (count_of_class + lambda) / (total_records + lambda * num_classes)
    """
    class_names = data['Class'].unique()
    class_count = len(class_names)
    total = len(data)
    return [
        {
            "class_title": c,
            "class_table": data[data['Class'] == c],
            "class_prob": ( (data['Class'] == c).sum() + laplace_lambda ) / ( total + laplace_lambda * class_count )
        }
        for c in class_names
    ]

def compute_conditional_probs(data, laplace_lambda=1.0):
    """
    Returns a dictionary of the form:
    {
      feature1: {
         value1: { classA: P(value1|classA), classB: P(value1|classB) },
         value2: { classA: ...,          classB: ... }
      },
      feature2: { ... }
    }
    using Laplace smoothing.
    """
    classes_info = get_class_info(data, laplace_lambda=laplace_lambda)
    cond_probs = {}

    for feat in features:
        cond_probs[feat] = {}
        # for each possible value of that feature
        for val in feature_values[feat]:
            cond_probs[feat][val] = {}
            # for each class
            for c_info in classes_info:
                c_title = c_info['class_title']
                c_table = c_info['class_table']
                c_count = c_table.shape[0]

                # how many times does (feat == val) appear in c_table?
                val_count = (c_table[feat] == val).sum()

                numerator = val_count + laplace_lambda
                denominator = c_count + laplace_lambda * len(feature_values[feat])

                cond_probs[feat][val][c_title] = numerator / denominator

    return cond_probs, classes_info

### Predicting and Evaluating Accuracy
Each row's predicted class is determined by:
\[
  \hat{y} = \arg\max_{c \in C} \Big(\log P(\text{class}=c) + \sum_{f \in \text{features}} \log P(f=\text{val}|c)\Big).
\]
Accuracy is the fraction of rows where the predicted class matches the actual class.

In [48]:
def predict_class(row, cond_probs, classes_info, laplace_lambda=0.1):
    """
    For a single row, compute log-probabilities for each class
    and return the class with the highest score.
    """
    # Store class -> sum(log(conditional probs) + log(prior))
    scores = {}

    for c_info in classes_info:
        c_title = c_info['class_title']
        # Start with log of prior
        score = log(c_info['class_prob'] + laplace_lambda, 2)

        for feat in features:
            # feature value in the current row
            val = row[feat]
            p_val = cond_probs[feat][val].get(c_title, laplace_lambda)
            score += log(p_val + laplace_lambda, 2)

        scores[c_title] = score

    # Return the class with the highest log-prob sum
    return max(scores, key=scores.get)

def determine_accuracy(train_data, test_data, laplace_lambda=0.1):
    """
    Train on train_data, then measure accuracy on test_data.
    """
    cond_probs, classes_info = compute_conditional_probs(train_data, laplace_lambda=laplace_lambda)

    correct = 0
    total = len(test_data)

    for idx, row in test_data.iterrows():
        predicted = predict_class(row, cond_probs, classes_info, laplace_lambda)
        if predicted == row['Class']:
            correct += 1

    return correct / total

## 4. K-Fold Cross-Validation (Manual and Stratified)
We implement 10-fold stratified cross-validation to estimate how well our model generalizes. For each fold:
1. Use that fold as a test set.
2. Concatenate the other 9 folds for training.
3. Compute accuracy on the test fold.
4. Collect all 10 accuracies.
We then report the **average accuracy** and **standard deviation**.

In [49]:
def stratified_k_folds(data, k=10):
    """
    Splits the data into k folds in a stratified manner.
    Returns a list of DataFrame folds.
    """
    data = data.sample(frac=1).reset_index(drop=True)
    folds = [[] for _ in range(k)]

    # Get classes and distribute each class among the k folds
    for c in data['Class'].unique():
        c_data = data[data['Class'] == c]
        c_data = c_data.sample(frac=1).reset_index(drop=True)

        c_size = len(c_data)
        fold_size = c_size // k
        remainder = c_size % k
        index_start = 0

        for i in range(k):
            size = fold_size + (1 if i < remainder else 0)
            subset = c_data.iloc[index_start:index_start + size].to_dict('records')
            folds[i].extend(subset)
            index_start += size

    # Convert each fold to DataFrame
    fold_dfs = [pd.DataFrame(fold).reset_index(drop=True) for fold in folds]
    return fold_dfs

def perform_k_fold_cross_validation(data, k=10, laplace_lambda=0.1):
    """
    Perform k-fold cross-validation in a stratified manner.
    Returns the list of accuracies.
    """
    folds = stratified_k_folds(data, k=k)
    accuracies = []

    for i in range(k):
        test_fold = folds[i]
        # train_fold is all other folds combined
        train_folds = folds[:i] + folds[i+1:]
        train_data = pd.concat(train_folds, ignore_index=True)

        accuracy = determine_accuracy(train_data, test_fold, laplace_lambda=laplace_lambda)
        accuracies.append(accuracy)

    return accuracies

## 5. Putting It All Together
We now:
1. Choose how to handle `?` (mode=0 or mode=1).
2. Set a Laplace smoothing parameter (e.g., `0.1`, `1.0`, `2.0`).
3. Split the data 80:20 for train-test.
4. Measure training accuracy (optional, or we can measure train-set accuracy on itself).
5. Perform 10-fold cross-validation on the **training** set.
6. Finally, measure accuracy on the 20% test set.

Let's pick some default parameters and see the results.

In [57]:
# 1) Decide missing-value mode
#    0 => '?' as third category
#    1 => replace '?' with most frequent value
missing_mode = 0  

# 2) Laplace smoothing parameter
laplace_lambda = 0.1

# Process the data for missing values
processed_df, feature_values = get_complete_data(df, mode=missing_mode)

# 3) Manual stratified split: 80% train, 20% test
train_df, test_df = stratified_train_test_split(processed_df, test_ratio=0.2)

# 4) Accuracy on the training set itself
train_accuracy_on_train = determine_accuracy(train_df, train_df, laplace_lambda=laplace_lambda)

# 5) 10-Fold Cross Validation on the training set
cv_accuracies = perform_k_fold_cross_validation(train_df, k=10, laplace_lambda=laplace_lambda)
cv_mean = statistics.mean(cv_accuracies)
cv_std = statistics.stdev(cv_accuracies)

# 6) Final test-set accuracy
test_accuracy = determine_accuracy(train_df, test_df, laplace_lambda=laplace_lambda)

# Print results
print(f"1. Train Set Accuracy (on itself): {train_accuracy_on_train * 100:.2f}%")
print("\n10-Fold Cross-Validation Results (on training set):")
for fold_idx, acc in enumerate(cv_accuracies, 1):
    print(f"  Fold {fold_idx}: {acc*100:.2f}%")
print(f"\n   Average Accuracy: {cv_mean*100:.2f}%")
print(f"   Standard Deviation: {cv_std*100:.2f}%")
print(f"\n2. Test Set Accuracy (20% unseen data): {test_accuracy * 100:.2f}%")

1. Train Set Accuracy (on itself): 89.91%

10-Fold Cross-Validation Results (on training set):
  Fold 1: 94.44%
  Fold 2: 80.56%
  Fold 3: 88.89%
  Fold 4: 80.00%
  Fold 5: 88.24%
  Fold 6: 94.12%
  Fold 7: 97.06%
  Fold 8: 91.18%
  Fold 9: 97.06%
  Fold 10: 85.29%

   Average Accuracy: 89.68%
   Standard Deviation: 6.26%

2. Test Set Accuracy (20% unseen data): 89.77%
